# Read-Write CSV file - Pyspark Review
- Notebook by Adam Lang
- Date: 9-1-2026

In [0]:
## read file -- below is the more graceful way to do this
## the other way is to use `.option("inferSchema", "true") \ `.option("header", "true")
df = spark.read.csv("/Volumes/workspace/default/orders_data/orders.csv",
                    header=True,
                    inferSchema=True)
display(df)

order_date,country,order_id,product,qty,price
2024-02-16,IN,1000,Shoes,2,54.37
2024-02-01,CA,1001,Backpack,4,77.01
2024-02-03,AU,1002,Jacket,1,95.07
2024-03-01,UK,1003,Jeans,1,42.03
2024-01-31,IN,1004,T-Shirt,3,15.94
2024-01-17,IN,1005,Watch,2,131.49
2024-01-15,UK,1006,Shoes,3,50.83
2024-01-18,IN,1007,Backpack,3,85.85
2024-01-25,AU,1008,T-Shirt,2,16.94
2024-01-06,IN,1009,Jeans,4,37.45


In [0]:
## schema
df.printSchema()

root
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)



## Custom Schema
- We can write custom schemas as we see below.

In [0]:
from pyspark.sql import functions as F, types as T

## set up csv schema
csv_schema = T.StructType([
    T.StructField("order_date", T.DateType()),
    T.StructField("country", T.StringType()),
    T.StructField("order_id", T.IntegerType()),
    T.StructField("product", T.StringType()),
    T.StructField("qty", T.IntegerType()),
    T.StructField("price", T.DoubleType()),
])

In [0]:
## now we use the schema
df = spark.read \
     .option("header", True) \
     .option("dateFormat", "yyyy-dd-MM") \
     .csv("/Volumes/workspace/default/orders_data/orders.csv", schema=csv_schema)


df.printSchema()
display(df)

root
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)



order_date,country,order_id,product,qty,price
null,IN,1000,Shoes,2,54.37
2024-01-02,CA,1001,Backpack,4,77.01
2024-03-02,AU,1002,Jacket,1,95.07
2024-01-03,UK,1003,Jeans,1,42.03
null,IN,1004,T-Shirt,3,15.94
null,IN,1005,Watch,2,131.49
null,UK,1006,Shoes,3,50.83
null,IN,1007,Backpack,3,85.85
null,AU,1008,T-Shirt,2,16.94
2024-06-01,IN,1009,Jeans,4,37.45


In [0]:
## view
df.display()

order_date,country,order_id,product,qty,price
null,IN,1000,Shoes,2,54.37
2024-01-02,CA,1001,Backpack,4,77.01
2024-03-02,AU,1002,Jacket,1,95.07
2024-01-03,UK,1003,Jeans,1,42.03
null,IN,1004,T-Shirt,3,15.94
null,IN,1005,Watch,2,131.49
null,UK,1006,Shoes,3,50.83
null,IN,1007,Backpack,3,85.85
null,AU,1008,T-Shirt,2,16.94
2024-06-01,IN,1009,Jeans,4,37.45


In [0]:
## another way to write code above
df = (spark.read
      .format("csv") 
      .options(header=True, dateFormat="yyyy-MM-dd")
      .schema(csv_schema)
      .load("/Volumes/workspace/default/orders_data/orders.csv"))

df.printSchema()
display(df)

root
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)



order_date,country,order_id,product,qty,price
2024-02-16,IN,1000,Shoes,2,54.37
2024-02-01,CA,1001,Backpack,4,77.01
2024-02-03,AU,1002,Jacket,1,95.07
2024-03-01,UK,1003,Jeans,1,42.03
2024-01-31,IN,1004,T-Shirt,3,15.94
2024-01-17,IN,1005,Watch,2,131.49
2024-01-15,UK,1006,Shoes,3,50.83
2024-01-18,IN,1007,Backpack,3,85.85
2024-01-25,AU,1008,T-Shirt,2,16.94
2024-01-06,IN,1009,Jeans,4,37.45


## Write df to Parquet 

In [0]:
## write df to parquet
# Create sample data since source CSV is unavailable
from pyspark.sql import types as T
from datetime import date

csv_schema = T.StructType([
    T.StructField("order_date", T.DateType()),
    T.StructField("country", T.StringType()),
    T.StructField("order_id", T.IntegerType()),
    T.StructField("product", T.StringType()),
    T.StructField("qty", T.IntegerType()),
    T.StructField("price", T.DoubleType()),
])

sample_data = [
    (date(2024, 2, 16), "IN", 1000, "Shoes", 2, 54.37),
    (date(2024, 2, 1), "CA", 1001, "Backpack", 4, 77.01),
    (date(2024, 2, 3), "AU", 1002, "Jacket", 1, 95.07),
    (date(2024, 3, 1), "UK", 1003, "Jeans", 1, 42.03),
    (date(2024, 1, 31), "IN", 1004, "T-Shirt", 3, 15.94)
]

df = spark.createDataFrame(sample_data, schema=csv_schema)

output_path = "/Volumes/workspace/default/orders_data/orders_parquet"

df.write \
  .mode("overwrite") \
  .parquet(output_path)

## read it back to verify
parquet_df = spark.read.parquet(output_path)
parquet_df.printSchema()
display(parquet_df)

root
 |-- order_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)



order_date,country,order_id,product,qty,price
2024-02-01,CA,1001,Backpack,4,77.01
2024-01-31,IN,1004,T-Shirt,3,15.94
2024-02-03,AU,1002,Jacket,1,95.07
2024-02-16,IN,1000,Shoes,2,54.37
2024-03-01,UK,1003,Jeans,1,42.03
